In [ ]:
import requests
import requests_cache
import json
import time
from tqdm import tqdm
from collections import defaultdict

def safe_request(url, params=None, retries=5, delay=2):
    for attempt in range(retries):
        try:
            response = SESSION.get(
                url,
                params=params,
                timeout=20,
                headers={
                    "User-Agent": "benchmark"
                }
            )

            if response.status_code != 200:
                print(f"HTTP {response.status_code}, retrying...")
                time.sleep(delay)
                continue

            if not response.text.strip():
                print("Empty response, retrying...")
                time.sleep(delay)
                continue

            return response.json()

        except Exception as e:
            print(f"Request error (attempt {attempt+1}): {e}")
            time.sleep(delay)

    return None

# --------------------------------------------------
# Enable HTTP caching (1 day)
# --------------------------------------------------
requests_cache.install_cache(
    "wikidata_cache",
    backend="memory",
    expire_after=86400
)

SESSION = requests.Session()

# --------------------------------------------------
# Step 1: Get Wikidata QIDs from Wikipedia titles
# Uses MediaWiki API (fast, batched)
# --------------------------------------------------
def get_wikidata_ids_from_wikipedia(titles, language="en", batch_size=50):

    title_to_qid = {}
    not_found = []

    endpoint = f"https://{language}.wikipedia.org/w/api.php"

    for i in tqdm(range(0, len(titles), batch_size), desc="Fetching QIDs"):
        batch = titles[i:i + batch_size]

        params = {
            "action": "query",
            "format": "json",
            "prop": "pageprops",
            "titles": "|".join(batch)
        }

        data = safe_request(endpoint, params=params)

        if not data:
            print("Skipping batch due to repeated failure.")
            continue

        pages = data.get("query", {}).get("pages", {})

        for page in pages.values():
            title = page.get("title")

            if "missing" in page:
                not_found.append(title)
                continue

            qid = page.get("pageprops", {}).get("wikibase_item")

            if qid:
                title_to_qid[title] = qid
            else:
                not_found.append(title)

        time.sleep(0.1)  # polite delay

    return title_to_qid, not_found


# --------------------------------------------------
# Step 2: Fetch statement counts in batches
# Uses wbgetentities (50 QIDs per request)
# --------------------------------------------------
def get_statement_counts(qids, batch_size=50):

    qid_to_count = {}
    endpoint = "https://www.wikidata.org/w/api.php"

    for i in tqdm(range(0, len(qids), batch_size), desc="Fetching statement counts"):
        batch = qids[i:i + batch_size]

        params = {
            "action": "wbgetentities",
            "format": "json",
            "ids": "|".join(batch),
            "props": "claims"
        }

        data = safe_request(endpoint, params=params)

        if not data:
            print("Skipping QID batch due to repeated failure.")
            continue

        entities = data.get("entities", {})

        for qid, entity in entities.items():
            claims = entity.get("claims", {})
            qid_to_count[qid] = len(claims)

        time.sleep(0.1)

    return qid_to_count


# --------------------------------------------------
# Step 3: Create percentile buckets
# --------------------------------------------------
def bucket_by_percentiles(results):
    """
    results: list of (title, qid, statement_count)
    sorted descending
    """
    total = len(results)
    top_cut = int(total * 0.33)
    mid_cut = int(total * 0.66)

    return {
        "66-100%": results[:top_cut],          # highest
        "33-66%": results[top_cut:mid_cut],
        "0-33%": results[mid_cut:]            # lowest
    }


# --------------------------------------------------
# Step 4: Main Pipeline
# --------------------------------------------------
def annotate_entities(input_path, output_path, language="en"):

    # Load original data
    with open(input_path, "r") as f:
        entities = json.load(f)

    titles = [item["title"] for item in entities]

    # ---- STEP 1: Wikipedia → Wikidata QIDs ----
    title_to_qid, not_found_titles = get_wikidata_ids_from_wikipedia(
        titles,
        language=language
    )

    print(f"\nFound QIDs: {len(title_to_qid)}")
    print(f"Not found: {len(not_found_titles)}")

    # ---- STEP 2: QIDs → Statement counts ----
    qids = list(title_to_qid.values())

    qid_to_count = get_statement_counts(qids)

    # ---- STEP 3: Prepare sortable list ----
    sortable_results = []

    for title, qid in title_to_qid.items():
        count = qid_to_count.get(qid, 0)
        sortable_results.append((title, qid, count))

    # Sort descending by statement count
    sortable_results.sort(key=lambda x: x[2], reverse=True)

    # ---- STEP 4: Create buckets ----
    buckets = bucket_by_percentiles(sortable_results)

    title_to_bucket = {}
    title_to_count = {}
    title_to_qid_final = {}

    for bucket_name, entries in buckets.items():
        for title, qid, count in entries:
            title_to_bucket[title] = bucket_name
            title_to_count[title] = count
            title_to_qid_final[title] = qid

    # Handle not found
    for title in not_found_titles:
        title_to_bucket[title] = "not_found"
        title_to_count[title] = 0
        title_to_qid_final[title] = None
    
    """
    # ---- STEP 5: Annotate original data (preserve order) ----
    for item in entities:
        title = item["title"]
        item["wikidata_id"] = title_to_qid_final.get(title)
        item["wikidata_statements"] = title_to_count.get(title, 0)
        item["popularity_bucket"] = title_to_bucket.get(title, "not_found")

    # ---- STEP 6: Save output ----
    with open(output_path, "w") as f:
        json.dump(entities, f, indent=2)

    print("\nDone. Annotated file saved.")
    """
    
    # ---- STEP 5: Build fully annotated list ----
    annotated_entities = []

    for item in entities:
        title = item["title"]

        item["wikidata_id"] = title_to_qid_final.get(title)
        item["wikidata_statements"] = title_to_count.get(title, 0)
        item["popularity_bucket"] = title_to_bucket.get(title, "not_found")

        annotated_entities.append(item)

    # ---- STEP 6: Sort globally by statement count (descending) ----
    annotated_entities.sort(
        key=lambda x: x["wikidata_statements"],
        reverse=True
    )

    # ---- STEP 7: Cluster by bucket ----
    clustered_output = {
        "66-100%": [],
        "33-66%": [],
        "0-33%": [],
        "not_found": []
    }

    for item in annotated_entities:
        bucket = item["popularity_bucket"]
        clustered_output[bucket].append(item)

    # ---- STEP 8: Save clustered + sorted output ----
    with open(output_path, "w") as f:
        json.dump(clustered_output, f, indent=2)

    print("\nDone. Sorted + clustered annotated file saved.")

# --------------------------------------------------
# =================== RUN ==========================
# --------------------------------------------------
if __name__ == "__main__":

    input_file = "./10000_random_entities_wikipedia.json"
    #output_file = "./10000_random_entities_wikipedia_popularity_unsorted.json"
    output_file = "./10000_random_entities_wikipedia_popularity_sorted_clustered.json"

    annotate_entities(input_file, output_file, language="en")

Fetching QIDs: 100%|██████████| 200/200 [00:59<00:00,  3.37it/s]



Found QIDs: 10000
Not found: 0


Fetching statement counts: 100%|██████████| 200/200 [03:44<00:00,  1.12s/it]


Done. Sorted + clustered annotated file saved.
